# Kaggle runner — HAM10000 experiments

**Setup (một lần cho mỗi notebook):**
1. *Add Input* → dataset `kmader/skin-cancer-mnist-ham10000`.
2. *Settings* → Accelerator **GPU T4 x2** hoặc **P100**; **Internet: On** (clone repo + tải trọng số ImageNet).
3. Sửa ô cấu hình bên dưới (danh sách thí nghiệm, seed), rồi *Save Version → Save & Run All (Commit)* để chạy nền.
4. Xong: tải `outputs.zip` ở tab *Output* về, giải nén vào `D:\DL\` (giữ cấu trúc `outputs/...`).

Test set **chỉ** được chấm khi `EVAL_TEST = True` cho cấu hình đã chốt (PLAN_BTL.md mục 0).

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL = "https://github.com/duty00/Deep-Learning.git"   # repo đang public nên clone không cần token
BRANCH = "main"
EXPERIMENTS = ["e02_concat_ce", "e07_film_la"]   # tên file trong configs/experiments/ (không có .json)
SEEDS = [42, 43, 44]
MICRO_BATCH = 32          # GPU 16 GB: 32 (accum 1). Batch hiệu dụng vẫn = 32.
EVAL_TEST = False         # True CHỈ khi cấu hình đã chốt bằng validation
DEBUG = False             # True: chạy thử 2 epoch x 8 batch

In [ ]:
import os, subprocess, glob
os.chdir("/kaggle/working")
if not os.path.exists("DL"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "DL"], check=True)
os.chdir("/kaggle/working/DL")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
# Kaggle đã có torch/torchvision/timm; chỉ cài package của nhóm, không đụng phiên bản torch.
subprocess.run("pip install -q -e . --no-deps && pip install -q timm", shell=True, check=True)

In [ ]:
# Liên kết dataset Kaggle vào đúng đường dẫn config mong đợi (data/raw/ham10000).
candidates = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)]
assert candidates, "Không thấy HAM10000_metadata.csv: hãy Add Input dataset kmader/skin-cancer-mnist-ham10000"
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/ham10000"):
    os.symlink(candidates[0], "data/raw/ham10000")
print("dataset:", candidates[0])
# Bắt buộc: xác nhận đúng file metadata + manifest đã commit (SHA-256, không rò rỉ lesion).
subprocess.run(["python", "scripts/make_splits.py", "--verify"], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import time, traceback

def run(args):
    print(">>>", " ".join(args), flush=True)
    subprocess.run(["python", "-m", "dl_project"] + args, check=True)

# Một run lỗi không làm mất các run còn lại (chạy nền nhiều giờ).
failed, t0 = [], time.time()
for exp in EXPERIMENTS:
    cfg = f"configs/experiments/{exp}.json"
    for seed in SEEDS:
        extra = ["--seed", str(seed), "--micro-batch-size", str(MICRO_BATCH)] + (["--debug"] if DEBUG else [])
        name = f"{exp}-s{seed}" + ("-debug" if DEBUG else "")
        try:
            run(["train", "--config", cfg] + extra)
            best = sorted(glob.glob(f"outputs/{name}/train/*/best.pt"))[-1]
            run(["evaluate", "--config", cfg, "--checkpoint", best, "--split", "validation"] + extra)
            if EVAL_TEST:
                run(["evaluate", "--config", cfg, "--checkpoint", best, "--split", "test"] + extra)
        except Exception:
            traceback.print_exc()
            failed.append(name)
        print(f"--- {name} xong | tổng thời gian {(time.time() - t0) / 60:.1f} phút", flush=True)
print("LỖI:" if failed else "Tất cả run thành công.", failed or "")

In [ ]:
subprocess.run(["python", "scripts/aggregate_results.py", "--split", "validation"], check=True)
print(open("reports/tables/results_validation.csv").read())
# Đóng gói kết quả (gồm checkpoint) để tải về máy.
subprocess.run("cd /kaggle/working/DL && zip -qr /kaggle/working/outputs.zip outputs reports/tables", shell=True, check=True)
print("-> /kaggle/working/outputs.zip")